In [ ]:
import pandas as pd
import math
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('endurance_motec_export.csv')

$$ P = VI \\
   E = \int_{t_0}^{t_1} P(t) dt
   \approx \sum_{i}P(i)\Delta t_i
$$

In [ ]:
def calcPowerUsage(df, t0=0, t1=float('inf')):
    filteredDf = df[(t1 >= df['Time']) & (df['Time'] >= t0)]

    #Factor of 100 as assuming both voltage and current are scaled by a factor of 10
    #So power is scaled by 100 (If curr data was in W car we be dq)
    filteredDf["Scaled Battery Power (W)"] = (filteredDf["Car Data Battery BatteryPower"] / 100)

    filteredDf["Delta Time (s)"] = filteredDf["Time"].diff().fillna(0)

    filteredDf["Power * dt"] = filteredDf["Scaled Battery Power (W)"] * filteredDf["Delta Time (s)"]
    #This is in (Ws)

    totalEnergy = filteredDf["Power * dt"].sum()

    #Conversion factor of Ws -> kWH
    CONVERSION = 3_600_600
    totalEnergyKWH = totalEnergy / CONVERSION

    print(f"Total Energy Usage = {totalEnergyKWH} kWh")
    return totalEnergyKWH



In [ ]:
calcPowerUsage(df)

$$ P_\text{mechanical}=\tau\frac{2\pi RPM}{60} = \frac{ \pi \tau RPM}{30} $$



In [ ]:
def calcEfficiency(df, t0=0, t1=float('inf')):
    filteredDf = df[(t1 >= df['Time']) & (df['Time'] >= t0)]

    #Factor of 100 as assuming both voltage and current are scaled by a factor of 10
    #So power is scaled by 100 (If curr data was in W car we be dq)
    filteredDf["Scaled Battery Power (W)"] = (filteredDf["Car Data Battery BatteryPower"] / 100)


    #Scaled by an additional factor of 10 to account for torque not being in Nm
    filteredDf["Mechanical Power"] =  abs(math.pi * filteredDf["Car Data Inverter InverterFDBTorque"] *  filteredDf["Car Data Motor MotorRPM"] / 300)

    #Ignoring when no (or little) power is being deployed
    filteredDf = filteredDf[(filteredDf['Scaled Battery Power (W)'] > 5000)].copy()

    filteredDf["Motor Efficiency"] = filteredDf["Mechanical Power"] / filteredDf["Scaled Battery Power (W)"]
   
    return filteredDf

In [ ]:
def createEfficiencyPlot(xName, yName, fDf):
    x = fDf[xName]
    y = fDf[yName]
    z = fDf["Motor Efficiency"]

    xbins = np.linspace(x.min(), x.max(), 10)
    ybins = np.linspace(y.min(), y.max(), 10)

    # Calculate mean efficiency in each rectangular bin
    sum_z, _, _ = np.histogram2d(x, y, bins=[xbins, ybins], weights=z)
    count, _, _ = np.histogram2d(x, y, bins=[xbins, ybins])

    mean_z = sum_z / count
    mean_z[count == 0] = np.nan

    plt.figure(figsize=(10, 7))

    plt.pcolormesh(
        xbins,
        ybins,
        mean_z.T,
        shading="auto",
        vmin=0,
        vmax=1.0
    )

    plt.colorbar(label="Motor Efficiency")

    plt.xlabel(xName)
    plt.ylabel(yName)
    plt.title(f"Motor Efficiency vs {xName} and {yName}")

    plt.show()

In [ ]:
fDf = calcEfficiency(df)

In [ ]:
createEfficiencyPlot("Car Data Battery HotSpotTemp",
"Car Data Motor MotorTemp", fDf)
createEfficiencyPlot("Car Data Motor MotorRPM",
"Car Data Battery BatteryPower", fDf)